In [ ]:
import os
import random
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset,TensorDataset,SubsetRandomSampler
from sklearn.metrics import classification_report
from torchvision import transforms
# from pywt import dwt2, idwt2, wavedec2
import torchvision.models as models
from sklearn.model_selection import train_test_split
import torch.nn.functional as F
from sklearn.metrics import precision_recall_fscore_support
import seaborn as sns
from sklearn.metrics import precision_recall_curve,f1_score,roc_auc_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import precision_recall_curve,f1_score,roc_curve,roc_auc_score,auc,accuracy_score,average_precision_score,precision_score,recall_score
from sklearn.preprocessing import label_binarize
import warnings
from sklearn.preprocessing import MinMaxScaler, StandardScaler
warnings.filterwarnings("ignore")
import torchvision
from torchvision import models
import torch.nn.functional as F
import torch.nn.init as init
from scipy import interp

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
def make_img(t_img):
    img = pd.read_pickle(t_img)
    img_l = []
    for i in range(len(img)):
        img_l.append(img.values[i][0])
    
    return np.array(img_l)

In [ ]:
def weight_init(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_normal_(m.weight)
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)
    elif isinstance(m, nn.Conv2d):
        nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.constant_(m.weight, 1)
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)
    elif isinstance(m, nn.LayerNorm):
        nn.init.constant_(m.weight, 1)
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)

In [ ]:
# # 设置随机种子
# def reset_random_seeds(seed):
#     torch.manual_seed(seed)
#     np.random.seed(seed)
#     random.seed(seed)
def set_random_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        # torch.cuda.manual_seed_all(seed)  # 如果使用多个GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
def plot_classification_report(y_tru, y_prd, mode, learning_rate, batch_size,epochs, figsize=(7, 7), ax=None):

    plt.figure(figsize=figsize)

    xticks = ['precision', 'recall', 'f1-score', 'support']
    yticks = ["Control", "Moderate", "Alzheimer's" ] 
    yticks += ['avg']

    rep = np.array(precision_recall_fscore_support(y_tru, y_prd)).T
    avg = np.mean(rep, axis=0)
    avg[-1] = np.sum(rep[:, -1])
    rep = np.insert(rep, rep.shape[0], avg, axis=0)

    sns.heatmap(rep,
                annot=True, 
                cbar=False, 
                xticklabels=xticks, 
                yticklabels=yticks,
                ax=ax, cmap = "Blues")
    
    plt.savefig('report_' + str(mode) + '_' + str(learning_rate) +'_' + str(batch_size)+'_' + str(epochs)+'.png')

In [ ]:
def calc_confusion_matrix(result, test_label,mode, learning_rate, batch_size, epochs):
    result = F.one_hot(result,num_classes=4)
    # print(result)

    test_label = F.one_hot(test_label,num_classes=4)
    # print(test_label)

    true_label= np.argmax(test_label, axis =1)

    predicted_label= np.argmax(result, axis =1)
    
    n_classes = 4
    precision = dict()
    recall = dict()
    thres = dict()
    for i in range(n_classes):
        precision[i], recall[i], thres[i] = precision_recall_curve(test_label[:, i],
                                                            result[:, i])


    print ("Classification Report :") 
    print (classification_report(true_label, predicted_label))
    cr = classification_report(true_label, predicted_label, output_dict=True)
    return cr, precision, recall, thres

In [ ]:
class create_modal_plasma(nn.Module):
    def __init__(self,transcriptomicsinput,num_classes=50):
        super(create_modal_plasma, self).__init__()
        self.fc1 = nn.Linear(transcriptomicsinput, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, num_classes)
        self.dropout1 = nn.Dropout(0.4)
        self.dropout2 = nn.Dropout(0.3)
        self.dropout3 = nn.Dropout(0.2)
        self.batch_norm1 = nn.BatchNorm1d(256)
        self.batch_norm2 = nn.BatchNorm1d(128)
        self.batch_norm3 = nn.BatchNorm1d(64)

    def forward(self, x):
        x = F.leaky_relu(self.fc1(x))
        x = self.batch_norm1(x)
        x = self.dropout1(x)
        x = F.leaky_relu(self.fc2(x))
        x = self.batch_norm2(x)
        x = self.dropout2(x)
        x = F.leaky_relu(self.fc3(x))
        x = self.batch_norm3(x)
        x = self.dropout3(x)
        x = self.fc4(x)
        return x

In [ ]:

class SELayer(nn.Module):
    def __init__(self, channel, reduction=16):
        super(SELayer, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channel, channel // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channel // reduction, channel, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)


# 多尺度卷积块
class MultiScaleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(MultiScaleConv, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(in_channels, out_channels, kernel_size=5, padding=2)
        self.conv5 = nn.Conv2d(in_channels, out_channels, kernel_size=7, padding=3)
        self.bn = nn.BatchNorm2d(out_channels * 3)

    def forward(self, x):
        x1 = self.conv1(x)
        x2 = self.conv3(x)
        x3 = self.conv5(x)
        x = torch.cat((x1, x2, x3), dim=1)
        x = self.bn(x)
        return x
    
class create_modal_img(nn.Module):
    def __init__(self, num_classes=10):
        super(create_modal_img, self).__init__()
        self.multi_scale_conv1 = MultiScaleConv(3, 64)
        self.se1 = SELayer(64 * 3)

        self.multi_scale_conv2 = MultiScaleConv(64 * 3, 100)
        self.se2 = SELayer(100 * 3)

        self.multi_scale_conv3 = MultiScaleConv(100 * 3, 50)
        self.se3 = SELayer(50 * 3)

        self.relu = nn.ReLU()
        self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.flatten = nn.Flatten()
        self.dropout = nn.Dropout(p=0.5)
        self.fc = nn.Linear(50 * 3 * 9 * 9, num_classes)  # 假设输入图像大小为 (3, 32, 32)

    def forward(self, x):
        x = self.relu(self.multi_scale_conv1(x))
        x = self.se1(x)
        x = self.maxpool(x)

        x = self.relu(self.multi_scale_conv2(x))
        x = self.se2(x)
        x = self.maxpool(x)

        x = self.relu(self.multi_scale_conv3(x))
        x = self.se3(x)
        x = self.maxpool(x)
        # print(x.shape)
        x = self.flatten(x)
        x = self.dropout(x)
        x = self.fc(x)
        return x
    
class new_AttentionMultiScaleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(new_AttentionMultiScaleCNN, self).__init__()
        self.multi_scale_conv1 = MultiScaleConv(3, 64)
        self.se1 = SELayer(64 * 3)

        self.multi_scale_conv2 = MultiScaleConv(64 * 3, 100)
        self.se2 = SELayer(100 * 3)

        self.multi_scale_conv3 = MultiScaleConv(100 * 3, 50)
        self.se3 = SELayer(50 * 3)

        self.relu = nn.ReLU()
        self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.flatten = nn.Flatten()
        self.dropout = nn.Dropout(p=0.5)
        self.fc = nn.Linear(50 * 3 * 9 * 9, num_classes)  # 假设输入图像大小为 (3, 32, 32)

    def forward(self, x):
        x = self.relu(self.multi_scale_conv1(x))
        x = self.se1(x)
        x = self.maxpool(x)

        x = self.relu(self.multi_scale_conv2(x))
        x = self.se2(x)
        x = self.maxpool(x)

        x = self.relu(self.multi_scale_conv3(x))
        x = self.se3(x)
        x = self.maxpool(x)
        # print(x.shape)
        x = self.flatten(x)
        x = self.dropout(x)
        x = self.fc(x)
        return x

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=100, kernel_size=3, stride=1)
        self.relu = nn.ReLU()
        self.maxpool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.dropout1 = nn.Dropout(p=0.5)
        self.conv2 = nn.Conv2d(in_channels=100, out_channels=50, kernel_size=3, stride=1)
        self.maxpool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.dropout2 = nn.Dropout(p=0.3)
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(50 * 16 * 16, 4)  # Assuming input shape (72, 72, 1) after convolutions and pooling

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.maxpool1(x)
        x = self.dropout1(x)
        x = self.relu(self.conv2(x))
        x = self.maxpool2(x)
        x = self.dropout2(x)
        x = self.flatten(x)
        x = self.fc(x)
        out=torch.softmax(x, dim=1)
        return out

In [ ]:

class NeuralNetwork(nn.Module):
    def __init__(self,input,num_classes=4):
        super(NeuralNetwork, self).__init__()
        self.fc1 = nn.Linear(input, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, num_classes)
        self.dropout1 = nn.Dropout(0.4)
        self.dropout2 = nn.Dropout(0.3)
        self.dropout3 = nn.Dropout(0.2)
        self.batch_norm1 = nn.BatchNorm1d(256)
        self.batch_norm2 = nn.BatchNorm1d(128)
        self.batch_norm3 = nn.BatchNorm1d(64)

    def forward(self, x):
        x = F.leaky_relu(self.fc1(x))
        x = self.batch_norm1(x)
        x = self.dropout1(x)
        x = F.leaky_relu(self.fc2(x))
        x = self.batch_norm2(x)
        x = self.dropout2(x)
        x = F.leaky_relu(self.fc3(x))
        x = self.batch_norm3(x)
        x = self.dropout3(x)
        x = self.fc4(x)
        return F.softmax(x, dim=1)

In [ ]:
class FC(nn.Module):
    def __init__(self, in_size, out_size, dropout_r=0., use_relu=True):
        super(FC, self).__init__()
        self.dropout_r = dropout_r
        self.use_relu = use_relu

        self.linear = nn.Linear(in_size, out_size)

        if use_relu:
            self.relu = nn.ReLU(inplace=True)

        if dropout_r > 0:
            self.dropout = nn.Dropout(dropout_r)

    def forward(self, x):
        x = self.linear(x)

        if self.use_relu:
            x = self.relu(x)

        if self.dropout_r > 0:
            x = self.dropout(x)

        return x

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_size, mid_size, out_size, dropout_r=0., use_relu=True):
        super(MLP, self).__init__()

        self.fc = FC(in_size, mid_size, dropout_r=dropout_r, use_relu=use_relu)
        self.linear = nn.Linear(mid_size, out_size)

    def forward(self, x):
        return self.linear(self.fc(x))

In [ ]:
class FFN(nn.Module):
    def __init__(self,):
        super(FFN, self).__init__()

        self.mlp = MLP(
            in_size=50,
            mid_size=100,
            out_size=50,
            dropout_r=0.3,
            use_relu=True
        )

    def forward(self, x):
        return self.mlp(x)

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self):
        super(SelfAttention, self).__init__()
        self.multihead_attn = nn.MultiheadAttention(embed_dim=50, num_heads=2)
        self.dropout1 = nn.Dropout(0.3)
        self.layer_norm1 = nn.LayerNorm(50)
        self.dropout2 = nn.Dropout(0.2)
        self.layer_norm2 = nn.LayerNorm(50)
        self.ffn=FFN()

    def forward(self, inputs):
        inputs=torch.unsqueeze(inputs, dim=1)
        outputs, _ = self.multihead_attn(inputs, inputs, inputs, attn_mask=None)
        outputs = self.dropout1(outputs)
        outputs = self.layer_norm1(inputs + outputs)
        outputs_2=self.ffn(outputs)
        outputs_2 = self.dropout2(outputs_2)
        all_outputs=self.layer_norm2(outputs + outputs_2)
        return all_outputs[:,0,:]
        # return outputs

In [ ]:
class Cross_modal_transformer(nn.Module):
    def __init__(self):
        super(Cross_modal_transformer, self).__init__()
        self.multihead_attn = nn.MultiheadAttention(embed_dim=50, num_heads=2)
        self.dropout1 = nn.Dropout(0.5)
        self.layer_norm1 = nn.LayerNorm(50)
        self.dropout2 = nn.Dropout(0.3)
        self.layer_norm2 = nn.LayerNorm(50)
        self.linear1=nn.Linear(50,50)
        self.linear2=nn.Linear(50,50)
        # self.ffn=FFN()

    def forward(self, x, y):
        x=torch.unsqueeze(x, dim=1)
        y=torch.unsqueeze(y, dim=1)

        for i in range(3):
            x = self.layer_norm1(x)
            y = self.layer_norm1(y)
            att = x + self.dropout1(self.multihead_attn(x, y, y, attn_mask=None)[0])

            # x = att + self.dropout2(self.linear2(F.relu(self.linear1(self.layer_norm2(att)))))
            x = att + self.dropout2(self.linear2(F.relu(self.linear1(self.layer_norm2(att)))))
        return x.squeeze(1)

In [ ]:
class multi_modal_model(nn.Module):
    def __init__(self,mode,transcriptomicsinput):
        super(multi_modal_model, self).__init__()
        self.plasma_modal=NeuralNetwork(transcriptomicsinput,num_classes=50)
        # self.plasma_modal.load_state_dict(torch.load('../../models/PPMI/Transcriptomics/best_146.pth'))
        # self.plasma_modal.fc4 = nn.Linear(64, 50)

        self.img_modal=new_AttentionMultiScaleCNN(num_classes=50)
        # self.img_modal.load_state_dict(torch.load('../../models/PPMI/MRI/best_22.pth'))
        # self.img_modal.fc = nn.Linear(50 * 3 * 9 * 9, 50)

        self.NeuralNetwork=NeuralNetwork(transcriptomicsinput,num_classes=4)
        self.img=new_AttentionMultiScaleCNN(num_classes=4)
    
        self.mode=mode


        self.fc0 = nn.Linear(100, 4)
        self.fc2 = nn.Linear(200,4)
        self.fc0.apply(weight_init)
        self.fc2.apply(weight_init)

        self.selfattention=SelfAttention()

        self.transformer=Cross_modal_transformer()
        self.transformer2=Cross_modal_transformer()
        self.transformer.apply(weight_init)
        self.transformer2.apply(weight_init)

        self.bn1=nn.BatchNorm1d(50)
        self.bn2=nn.BatchNorm1d(50)
        self.bn1.apply(weight_init)
        self.bn2.apply(weight_init)

    def forward(self, train_plasma, train_img):

        dense_plasma=self.bn1(self.plasma_modal(train_plasma))
        dense_img=self.bn2(self.img_modal(train_img))
        if self.mode == 'None':

            merged = torch.cat((dense_plasma,dense_img),axis=1)

            output = self.fc0(merged)
            return dense_plasma,dense_img,torch.softmax(output,dim=1)
        elif self.mode == 'MM_Tansformer':
    
            pp_att = self.transformer(dense_plasma,dense_img)
            ii_att = self.transformer2(dense_img,dense_plasma)
            
            merged = torch.cat((pp_att,ii_att),axis=1)
            output=self.fc0(merged)
            return pp_att,ii_att,torch.softmax(output,dim=1)
        # return torch.softmax(output,dim=1)

In [ ]:
class multi_modal_model_OGM(nn.Module):
    def __init__(self,mode,transcriptomicsinput):
        super(multi_modal_model_OGM, self).__init__()
        self.plasma_modal=NeuralNetwork(transcriptomicsinput,num_classes=4)
        self.plasma_modal.load_state_dict(torch.load('../../models/PPMI/Transcriptomics/best_146.pth'))
        self.plasma_modal.fc4 = nn.Linear(64, 50)

        self.img_modal=new_AttentionMultiScaleCNN(num_classes=4)
        self.img_modal.load_state_dict(torch.load('../../models/PPMI/MRI/best_22.pth'))
        self.img_modal.fc = nn.Linear(50 * 3 * 9 * 9, 50)

        self.NeuralNetwork=NeuralNetwork(transcriptomicsinput,num_classes=4)
        self.img=new_AttentionMultiScaleCNN(num_classes=4)
    
        self.mode=mode


        self.fc0 = nn.Linear(100, 4)
        self.fc2 = nn.Linear(200,4)
        self.fc0.apply(weight_init)
        self.fc2.apply(weight_init)

        self.selfattention=SelfAttention()

        self.transformer=Cross_modal_transformer()
        self.transformer2=Cross_modal_transformer()
        self.transformer.apply(weight_init)
        self.transformer2.apply(weight_init)

        self.bn1=nn.BatchNorm1d(50)
        self.bn2=nn.BatchNorm1d(50)
        self.bn1.apply(weight_init)
        self.bn2.apply(weight_init)

    def forward(self, train_plasma, train_img):

        dense_plasma=self.bn1(self.plasma_modal(train_plasma))
        dense_img=self.bn2(self.img_modal(train_img))
        if self.mode == 'None':

            merged = torch.cat((dense_plasma,dense_img),axis=1)

            output = self.fc0(merged)
            return dense_plasma,dense_img,torch.softmax(output,dim=1)
        elif self.mode == 'MM_Tansformer':
    
            pp_att = self.transformer(dense_plasma,dense_img)
            ii_att = self.transformer2(dense_img,dense_plasma)
            
            merged = torch.cat((pp_att,ii_att),axis=1)
            output=self.fc0(merged)
            return pp_att,ii_att,torch.softmax(output,dim=1)
        # return torch.softmax(output,dim=1)

In [ ]:
test_transcriptomics = pd.read_csv("../../processed_data/PPMI/overlap/X_test_transpot.csv").drop("PATNO Visit", axis=1).values
test_img = make_img("../../processed_data/PPMI/overlap/X_test_img.pkl")
test_label = pd.read_csv("../../processed_data/PPMI/overlap/y_test.csv").drop("PATNO Visit", axis=1).values.astype("int").flatten()
test_transcriptomics_tensor = torch.tensor(test_transcriptomics, dtype=torch.float).to(device)
test_img_tensor = torch.tensor(test_img, dtype=torch.float).to(device)

In [ ]:
# plt.figure(figsize=(10, 8))
# num_classes=4
# colors = ['#FF7F00', '#A5C2E2', '#FFE0C1','#70B48F','#5861AC']#
# lw = 2
# model_list=[]

# transpot_modal=NeuralNetwork(test_transcriptomics.shape[1]).to(device)
# transpot_modal.load_state_dict(torch.load(f'../../models/PPMI/Transcriptomics/best_146.pth'))
# transpot_modal.eval()
# model_list.append(transpot_modal)

# img_modal=new_AttentionMultiScaleCNN(num_classes=4).to(device)
# img_modal.load_state_dict(torch.load('../../models/PPMI/MRI/best_22.pth'))
# img_modal.eval()
# model_list.append(img_modal)

# Concate_modal=multi_modal_model('None',test_transcriptomics.shape[1]).to(device)
# Concate_modal.load_state_dict(torch.load(f'../../models/PPMI/Multi-modal/None_best_84.pth'))
# Concate_modal.eval()
# model_list.append(Concate_modal)

# # Concate_OGM_modal=multi_modal_model('None',test_transcriptomics.shape[1]).to(device)
# # Concate_OGM_modal.load_state_dict(torch.load(f'../../models/PPMI/Multi-modal/None_OGM_0_best_19.pth'))
# # Concate_OGM_modal.eval()
# # model_list.append(Concate_OGM_modal)

# Cross_modal_transformer_modal=multi_modal_model('MM_Tansformer',test_transcriptomics.shape[1]).to(device)
# Cross_modal_transformer_modal.load_state_dict(torch.load(f'../../models/PPMI/Multi-modal/2fc/MM_Tansformer_best_179.pth'))
# Cross_modal_transformer_modal.eval()
# model_list.append(Cross_modal_transformer_modal)

# Cross_modal_transformer_OGM_modal=multi_modal_model('MM_Tansformer',test_transcriptomics.shape[1]).to(device)
# Cross_modal_transformer_OGM_modal.load_state_dict(torch.load(f'../../models/PPMI/Multi-modal/MM_Tansformer_OGM_0.2_best_152.pth'))
# Cross_modal_transformer_OGM_modal.eval()
# model_list.append(Cross_modal_transformer_OGM_modal)

# true_label_binarized = label_binarize(test_label, classes=list(range(num_classes)))

# model_name=['Transcriptomics','MRI','Concate','Cross_modal_transformer','Cross_modal_transformer_OGM']
# for model_num in range(len(model_list)):
#     n_classes = 4
#     predicted_label=[]
#     with torch.no_grad():
#         if model_num==0:
#             outputs = model_list[0](test_transcriptomics_tensor)
#         elif model_num==1:
#             outputs = model_list[1](test_img_tensor)
#         else:
#             _,_,outputs = model_list[model_num](test_transcriptomics_tensor,test_img_tensor)
                
#         _, predicted = outputs.max(1)
#         predicted_label.extend(predicted.tolist())
#     # 计算每一类的ROC
#     fpr = dict()
#     tpr = dict()
#     roc_auc = dict()
#     for i in range(n_classes):
#         fpr[i], tpr[i], _ = roc_curve(true_label_binarized[:, i], outputs.cpu().detach().numpy()[:, i])
#         roc_auc[i] = auc(fpr[i], tpr[i])
    
#     all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
#     # Then interpolate all ROC curves at this points
#     mean_tpr = np.zeros_like(all_fpr)
#     for i in range(n_classes):
#         mean_tpr += interp(all_fpr, fpr[i], tpr[i])

#     mean_tpr /= n_classes
#     fpr["macro"] = all_fpr
#     tpr["macro"] = mean_tpr
#     roc_auc["macro"] = auc(fpr["macro"], tpr["macro"],)

#     # Plot all ROC curves
#     lw=2
#     plt.plot(fpr["macro"], tpr["macro"],
#             label=f'{model_name[model_num]}'+' (area = {0:0.2f})'
#                 ''.format(roc_auc["macro"]),
#             color=colors[model_num], linestyle='-', linewidth=2)
 
# plt.plot([0, 1], [0, 1], 'k--', lw=lw)
# plt.xlim([0.0, 1.0])
# plt.ylim([0.0, 1.05])
# plt.xlabel('False Positive Rate',size=12)
# plt.ylabel('True Positive Rate',size=12)
# # plt.title('PPMI ROC')
# plt.legend(loc="lower right")
# # plt.savefig('../../figure/PPMI/PPMI_ROC.png',dpi=300)

In [ ]:
# y = label_binarize(test_label, classes=[0, 1, 2, 3])
# plt.figure(figsize=(10, 8))


# num_classes=4
# colors = ['#FF7F00', '#A5C2E2', '#FFE0C1','#70B48F','#5861AC']#
# lw = 2
# model_list=[]

# transpot_modal=NeuralNetwork(test_transcriptomics.shape[1]).to(device)
# transpot_modal.load_state_dict(torch.load(f'../../models/PPMI/Transcriptomics/best_146.pth'))
# transpot_modal.eval()
# model_list.append(transpot_modal)

# img_modal=new_AttentionMultiScaleCNN(num_classes=4).to(device)
# img_modal.load_state_dict(torch.load('../../models/PPMI/MRI/best_22.pth'))
# img_modal.eval()
# model_list.append(img_modal)

# Concate_modal=multi_modal_model('None',test_transcriptomics.shape[1]).to(device)
# Concate_modal.load_state_dict(torch.load(f'../../models/PPMI/Multi-modal/None_best_84.pth'))
# Concate_modal.eval()
# model_list.append(Concate_modal)

# # Concate_OGM_modal=multi_modal_model('None',test_transcriptomics.shape[1]).to(device)
# # Concate_OGM_modal.load_state_dict(torch.load(f'../../models/PPMI/Multi-modal/None_OGM_0_best_19.pth'))
# # Concate_OGM_modal.eval()
# # model_list.append(Concate_OGM_modal)

# Cross_modal_transformer_modal=multi_modal_model('MM_Tansformer',test_transcriptomics.shape[1]).to(device)
# Cross_modal_transformer_modal.load_state_dict(torch.load(f'../../models/PPMI/Multi-modal/2fc/MM_Tansformer_best_179.pth'))
# Cross_modal_transformer_modal.eval()
# model_list.append(Cross_modal_transformer_modal)

# Cross_modal_transformer_OGM_modal=multi_modal_model('MM_Tansformer',test_transcriptomics.shape[1]).to(device)
# Cross_modal_transformer_OGM_modal.load_state_dict(torch.load(f'../../models/PPMI/Multi-modal/MM_Tansformer_OGM_0.2_best_152.pth'))
# Cross_modal_transformer_OGM_modal.eval()
# model_list.append(Cross_modal_transformer_OGM_modal)

# true_label_binarized = label_binarize(test_label, classes=list(range(num_classes)))
# for model_num in range(len(model_list)):
#     with torch.no_grad():
#         if model_num==0:
#             outputs = model_list[0](test_transcriptomics_tensor)
#         elif model_num==1:
#             outputs = model_list[1](test_img_tensor)
#         else:
#             _,_,outputs = model_list[model_num](test_transcriptomics_tensor,test_img_tensor)
#     precision = dict()
#     recall = dict()
#     average_precision = dict()
#     for i in range(n_classes):
#         precision[i], recall[i], _ = precision_recall_curve(y[:, i], outputs.cpu()[:, i])
#         average_precision[i] = average_precision_score(y[:, i], outputs.cpu()[:, i])

#     # 计算宏平均PR曲线
#     all_precision = np.unique(np.concatenate([precision[i] for i in range(n_classes)]))
#     mean_recall = np.zeros_like(all_precision)
#     for i in range(n_classes):
#         mean_recall += np.interp(all_precision, precision[i], recall[i])

#     mean_recall /= n_classes

#     # 计算宏平均AUPR
#     mean_aupr = average_precision_score(y, outputs.cpu(), average="macro")

#     # 绘制宏平均PR曲线

#     plt.plot(mean_recall, all_precision, color=colors[model_num], lw=2,
#             label=f'{model_name[model_num]}'+' (area = {0:0.2f})'
#             ''.format(mean_aupr),linestyle='-')
# plt.xlabel('Recall',size=12)
# plt.ylabel('Precision',size=12)
# # plt.title('Precision-Recall curve for multi-class')
# plt.legend(loc="best")
# # plt.savefig('../../figure/PPMI/PPMI_PR.png',dpi=300)

In [ ]:
plt.figure(figsize=(10, 8))
num_classes=4
colors = ['#FF7F00', '#A5C2E2', '#FFE0C1','#70B48F','#5861AC']#
lw = 2
model_list=[]

transpot_modal=NeuralNetwork(test_transcriptomics.shape[1]).to(device)
transpot_modal.load_state_dict(torch.load(f'../../models/PPMI/Transcriptomics/best_146.pth'))
transpot_modal.eval()
model_list.append(transpot_modal)

img_modal=new_AttentionMultiScaleCNN(num_classes=4).to(device)
img_modal.load_state_dict(torch.load('../../models/PPMI/MRI/best_22.pth'))
img_modal.eval()
model_list.append(img_modal)

Concate_modal=multi_modal_model('None',test_transcriptomics.shape[1]).to(device)
Concate_modal.load_state_dict(torch.load(f'../../models2/PPMI/Multi-modal/None_best_11.pth'))
Concate_modal.eval()
model_list.append(Concate_modal)

# Concate_OGM_modal=multi_modal_model('None',test_transcriptomics.shape[1]).to(device)
# Concate_OGM_modal.load_state_dict(torch.load(f'../../models/PPMI/Multi-modal/None_OGM_0_best_19.pth'))
# Concate_OGM_modal.eval()
# model_list.append(Concate_OGM_modal)

Cross_modal_transformer_modal=multi_modal_model('MM_Tansformer',test_transcriptomics.shape[1]).to(device)
Cross_modal_transformer_modal.load_state_dict(torch.load(f'../../models2/PPMI/Multi-modal/MM_Tansformer_best_42.pth'))
Cross_modal_transformer_modal.eval()
model_list.append(Cross_modal_transformer_modal)

Cross_modal_transformer_OGM_modal=multi_modal_model('MM_Tansformer',test_transcriptomics.shape[1]).to(device)
Cross_modal_transformer_OGM_modal.load_state_dict(torch.load(f'../../models2/PPMI/Multi-modal/MM_Tansformer_OGM_0.2_best_167.pth'))
Cross_modal_transformer_OGM_modal.eval()
model_list.append(Cross_modal_transformer_OGM_modal)

true_label_binarized = label_binarize(test_label, classes=list(range(num_classes)))

model_name=['Transcriptomics','MRI','Concate','Cross_modal_transformer','Cross_modal_transformer_OGM']
for model_num in range(len(model_list)):
    n_classes = 4
    predicted_label=[]
    with torch.no_grad():
        if model_num==0:
            outputs = model_list[0](test_transcriptomics_tensor)
        elif model_num==1:
            outputs = model_list[1](test_img_tensor)
        else:
            _,_,outputs = model_list[model_num](test_transcriptomics_tensor,test_img_tensor)
                
        _, predicted = outputs.max(1)
        predicted_label.extend(predicted.tolist())
    # 计算每一类的ROC
    fpr = dict()
    tpr = dict()
    roc_auc = dict()
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(true_label_binarized[:, i], outputs.cpu().detach().numpy()[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
    
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
    # Then interpolate all ROC curves at this points
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_classes):
        mean_tpr += interp(all_fpr, fpr[i], tpr[i])

    mean_tpr /= n_classes
    fpr["macro"] = all_fpr
    tpr["macro"] = mean_tpr
    roc_auc["macro"] = auc(fpr["macro"], tpr["macro"],)

    # Plot all ROC curves
    lw=2
    plt.plot(fpr["macro"], tpr["macro"],
            label=f'{model_name[model_num]}'+' (area = {0:0.2f})'
                ''.format(roc_auc["macro"]),
            color=colors[model_num], linestyle='-', linewidth=2)
 
plt.plot([0, 1], [0, 1], 'k--', lw=lw)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate',size=12)
plt.ylabel('True Positive Rate',size=12)
# plt.title('PPMI ROC')
plt.legend(loc="lower right")
# plt.savefig('../../figure/PPMI/PPMI_ROC.png',dpi=300)

In [ ]:
y = label_binarize(test_label, classes=[0, 1, 2, 3])
plt.figure(figsize=(10, 8))


num_classes=4
colors = ['#FF7F00', '#A5C2E2', '#FFE0C1','#70B48F','#5861AC']#
lw = 2
model_list=[]

transpot_modal=NeuralNetwork(test_transcriptomics.shape[1]).to(device)
transpot_modal.load_state_dict(torch.load(f'../../models/PPMI/Transcriptomics/best_146.pth'))
transpot_modal.eval()
model_list.append(transpot_modal)

img_modal=new_AttentionMultiScaleCNN(num_classes=4).to(device)
img_modal.load_state_dict(torch.load('../../models/PPMI/MRI/best_22.pth'))
img_modal.eval()
model_list.append(img_modal)

Concate_modal=multi_modal_model('None',test_transcriptomics.shape[1]).to(device)
Concate_modal.load_state_dict(torch.load(f'../../models2/PPMI/Multi-modal/None_best_11.pth'))
Concate_modal.eval()
model_list.append(Concate_modal)

# Concate_OGM_modal=multi_modal_model('None',test_transcriptomics.shape[1]).to(device)
# Concate_OGM_modal.load_state_dict(torch.load(f'../../models/PPMI/Multi-modal/None_OGM_0_best_19.pth'))
# Concate_OGM_modal.eval()
# model_list.append(Concate_OGM_modal)

Cross_modal_transformer_modal=multi_modal_model('MM_Tansformer',test_transcriptomics.shape[1]).to(device)
Cross_modal_transformer_modal.load_state_dict(torch.load(f'../../models2/PPMI/Multi-modal/MM_Tansformer_best_42.pth'))
Cross_modal_transformer_modal.eval()
model_list.append(Cross_modal_transformer_modal)

Cross_modal_transformer_OGM_modal=multi_modal_model('MM_Tansformer',test_transcriptomics.shape[1]).to(device)
Cross_modal_transformer_OGM_modal.load_state_dict(torch.load(f'../../models2/PPMI/Multi-modal/MM_Tansformer_OGM_0.2_best_167.pth'))
Cross_modal_transformer_OGM_modal.eval()
model_list.append(Cross_modal_transformer_OGM_modal)

true_label_binarized = label_binarize(test_label, classes=list(range(num_classes)))
for model_num in range(len(model_list)):
    with torch.no_grad():
        if model_num==0:
            outputs = model_list[0](test_transcriptomics_tensor)
        elif model_num==1:
            outputs = model_list[1](test_img_tensor)
        else:
            _,_,outputs = model_list[model_num](test_transcriptomics_tensor,test_img_tensor)
    precision = dict()
    recall = dict()
    average_precision = dict()
    for i in range(n_classes):
        precision[i], recall[i], _ = precision_recall_curve(y[:, i], outputs.cpu()[:, i])
        average_precision[i] = average_precision_score(y[:, i], outputs.cpu()[:, i])

    # 计算宏平均PR曲线
    all_precision = np.unique(np.concatenate([precision[i] for i in range(n_classes)]))
    mean_recall = np.zeros_like(all_precision)
    for i in range(n_classes):
        mean_recall += np.interp(all_precision, precision[i], recall[i])

    mean_recall /= n_classes

    # 计算宏平均AUPR
    mean_aupr = average_precision_score(y, outputs.cpu(), average="macro")

    # 绘制宏平均PR曲线

    plt.plot(mean_recall, all_precision, color=colors[model_num], lw=2,
            label=f'{model_name[model_num]}'+' (area = {0:0.2f})'
            ''.format(mean_aupr),linestyle='-')
plt.xlabel('Recall',size=12)
plt.ylabel('Precision',size=12)
# plt.title('Precision-Recall curve for multi-class')
plt.legend(loc="best")
# plt.savefig('../../figure/PPMI/PPMI_PR.png',dpi=300)

In [ ]:
y = label_binarize(test_label, classes=[0, 1, 2, 3])

num_classes=4
colors = ['#FF7F00', '#A5C2E2', '#FFE0C1','#70B48F','#5861AC']#
lw = 2
model_list=[]

Concate_modal=multi_modal_model('None',test_transcriptomics.shape[1]).to(device)
Concate_modal.load_state_dict(torch.load(f'../../models/PPMI/Multi-modal/None_best_84.pth'))
Concate_modal.eval()
model_list.append(Concate_modal)

Concate_OGM_modal=multi_modal_model('None',test_transcriptomics.shape[1]).to(device)
Concate_OGM_modal.load_state_dict(torch.load(f'../../models/PPMI/Multi-modal/None_OGM_0.2_best_78.pth'))
Concate_OGM_modal.eval()
model_list.append(Concate_OGM_modal)

Cross_modal_transformer_modal=multi_modal_model('MM_Tansformer',test_transcriptomics.shape[1]).to(device)
Cross_modal_transformer_modal.load_state_dict(torch.load(f'../../models/PPMI/Multi-modal/2fc/MM_Tansformer_best_179.pth'))
Cross_modal_transformer_modal.eval()
model_list.append(Cross_modal_transformer_modal)

Cross_modal_transformer_OGM_modal=multi_modal_model('MM_Tansformer',test_transcriptomics.shape[1]).to(device)
Cross_modal_transformer_OGM_modal.load_state_dict(torch.load(f'../../models/PPMI/Multi-modal/MM_Tansformer_OGM_0.2_best_152.pth'))
Cross_modal_transformer_OGM_modal.eval()
model_list.append(Cross_modal_transformer_OGM_modal)

true_label_binarized = label_binarize(test_label, classes=list(range(num_classes)))
model_name=['Concate','Concate_OGM','Cross_modal_transformer','Cross_modal_transformer_OGM']
print('PPMI')
for model_num in range(len(model_list)):
    with torch.no_grad():
        dense_transpot,dense_img,outputs = model_list[model_num](test_transcriptomics_tensor,test_img_tensor)
        cosine_similarity = F.cosine_similarity(dense_transpot, dense_img, dim=1)
        mean_cosine_similarity = cosine_similarity.mean()
        print(model_name[model_num],"Mean Cosine Similarity:", mean_cosine_similarity.item())
        # euclidean_distance = torch.norm(dense_transpot - dense_img, dim=1)
        # mean_euclidean_distance = euclidean_distance.mean()

        # print("Mean Euclidean Distance:", mean_euclidean_distance.item())
        # manhattan_distance = torch.sum(torch.abs(dense_transpot - dense_img), dim=1)
        # mean_manhattan_distance = manhattan_distance.mean()

        # print("Mean Manhattan Distance:", mean_manhattan_distance.item())
        # mse_loss = F.mse_loss(dense_transpot, dense_img)
        # print("Mean Squared Error:", mse_loss.item())

In [ ]:
Concate_modal=multi_modal_model('None',test_transcriptomics.shape[1]).to(device)
Concate_modal.load_state_dict(torch.load(f'../../models2/PPMI/Multi-modal/None_best_11.pth'))
Concate_modal.eval()
predicted_label_0=[]
with torch.no_grad():
    dense_plasma,dense_img,outputs_0 = Concate_modal(test_transcriptomics_tensor,test_img_tensor)
    # probs_0 = nn.functional.softmax(outputs_0, dim=1)
    # all_probs_0.extend(probs_0.cpu().detach().numpy())
    _, predicted_0 = outputs_0.max(1)
    predicted_label_0.extend(predicted_0.tolist())

In [ ]:
print(dense_plasma.shape,dense_img.shape)

In [ ]:
from sklearn.decomposition import PCA

In [ ]:
pca = PCA(n_components=2)
mri_pca = pca.fit_transform(dense_img.cpu())
transcriptomics_pca = pca.fit_transform(dense_plasma.cpu())

In [ ]:
plt.subplot(1, 1, 1)
plt.scatter(mri_pca[:, 0], mri_pca[:, 1], alpha=0.7, c='blue',label='MRI')
plt.scatter(transcriptomics_pca[:, 0], transcriptomics_pca[:, 1], alpha=0.7, c='red',label='transcriptomics')
# plt.title('MRI Data PCA')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
# 转录组数据的PCA可视化
# plt.subplot(1, 2, 2)

# plt.title('Transcriptomics Data PCA')
# plt.xlabel('Principal Component 1')
# plt.ylabel('Principal Component 2')
plt.legend()
plt.tight_layout()
# plt.savefig('../../figure/PPMI/PPMI_distribution.png',dpi=300)
# plt.show()

In [ ]:
plt.subplot(1, 2, 1)
plt.scatter(mri_pca[:, 0], mri_pca[:, 1], alpha=0.7, c='blue',label='MRI')
plt.title('MRI Data PCA')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
# 转录组数据的PCA可视化
plt.subplot(1, 2, 2)
plt.scatter(transcriptomics_pca[:, 0], transcriptomics_pca[:, 1], alpha=0.7, c='red',label='transcriptomics')
plt.title('Transcriptomics Data PCA')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
# plt.legend()
plt.tight_layout()
# plt.savefig('../../figure/PPMI/PPMI_distribution2.png',dpi=300)
# plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(pd.DataFrame(dense_img.cpu()).corr(), cmap='coolwarm', center=0)
plt.title('MRI Data Correlation Heatmap')
# plt.savefig('../../figure/PPMI/PPMI_MRI_correlation.png',dpi=300)
# plt.show()

plt.figure(figsize=(10, 8))
sns.heatmap(pd.DataFrame(dense_plasma.cpu()).corr(), cmap='coolwarm', center=0)
plt.title('Transcriptomics Data Correlation Heatmap')
# plt.savefig('../../figure/PPMI/PPMI_Transcriptomics_correlation.png',dpi=300)
# plt.show()

In [ ]:
from sklearn.cluster import KMeans
import seaborn as sns

# 聚类分析
kmeans = KMeans(n_clusters=4, random_state=42)
mri_clusters = kmeans.fit_predict(mri_pca)
transcriptomics_clusters = kmeans.fit_predict(transcriptomics_pca)

# 可视化聚类结果
# plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
sns.scatterplot(x=mri_pca[:, 0], y=mri_pca[:, 1], hue=mri_clusters, palette='viridis')
plt.title('MRI Data Clustering')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')

plt.subplot(1, 2, 2)
sns.scatterplot(x=transcriptomics_pca[:, 0], y=transcriptomics_pca[:, 1], hue=transcriptomics_clusters, palette='viridis')
plt.title('Transcriptomics Data Clustering')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')

plt.tight_layout()
plt.show()


In [ ]:
from sklearn.decomposition import FactorAnalysis

# 因子分析
fa = FactorAnalysis(n_components=2)
mri_fa = fa.fit_transform(dense_img.cpu())
transcriptomics_fa = fa.fit_transform(dense_plasma.cpu())

# 可视化因子分析结果
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.scatter(mri_fa[:, 0], mri_fa[:, 1], alpha=0.7, c='blue')
plt.title('MRI Data Factor Analysis')
plt.xlabel('Factor 1')
plt.ylabel('Factor 2')

plt.subplot(1, 2, 2)
plt.scatter(transcriptomics_fa[:, 0], transcriptomics_fa[:, 1], alpha=0.7, c='red')
plt.title('Transcriptomics Data Factor Analysis')
plt.xlabel('Factor 1')
plt.ylabel('Factor 2')

plt.tight_layout()
plt.show()


In [ ]:
from sklearn.cross_decomposition import CCA

# 联合分析
cca = CCA(n_components=2)
mri_cca, transcriptomics_cca = cca.fit_transform(dense_img.cpu(), dense_plasma.cpu())

# 可视化联合分析结果
# plt.figure(figsize=(12, 6))

plt.subplot(1, 1, 1)
plt.scatter(mri_cca[:, 0], mri_cca[:, 1], alpha=0.7, c='blue',label='MRI')
plt.scatter(transcriptomics_cca[:, 0], transcriptomics_cca[:, 1], alpha=0.7, c='red',label='Transcriptomics')
# plt.title('MRI Data CCA')
# plt.xlabel('Canonical Component 1')
# plt.ylabel('Canonical Component 2')

# plt.subplot(1, 2, 2)

# plt.title('Transcriptomics Data CCA')
# plt.xlabel('Canonical Component 1')
# plt.ylabel('Canonical Component 2')
plt.legend()
plt.tight_layout()
# plt.show()
# plt.savefig('../../figure/PPMI/PPMI_CCA.png',dpi=300)


In [ ]:
from sklearn.cross_decomposition import CCA

# 联合分析
cca = CCA(n_components=2)
mri_cca, transcriptomics_cca = cca.fit_transform(dense_img.cpu(), dense_plasma.cpu())

# 可视化联合分析结果
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.scatter(mri_cca[:, 0], mri_cca[:, 1], alpha=0.7, c='blue')
plt.title('MRI Data CCA')
plt.xlabel('Canonical Component 1')
plt.ylabel('Canonical Component 2')

plt.subplot(1, 2, 2)
plt.scatter(transcriptomics_cca[:, 0], transcriptomics_cca[:, 1], alpha=0.7, c='red')
plt.title('Transcriptomics Data CCA')
plt.xlabel('Canonical Component 1')
plt.ylabel('Canonical Component 2')

plt.tight_layout()
# plt.show()
# plt.savefig('../../figure/PPMI/PPMI_CCA2.png',dpi=300)
